# Date Frequency Analysis

Analyzing the number of applications per day from the GDG applications database.

In [ ]:
# Import required libraries
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime

In [ ]:
# Connect to the database
db_path = '../../db/applications.db'
conn = sqlite3.connect(db_path)

# Load data from applications table
query = "SELECT Timestamp FROM applications"
df = pd.read_sql_query(query, conn)

print(f"Total records: {len(df)}")
print("\nSample timestamps:")
print(df['Timestamp'].head(10))

In [ ]:
# Clean and parse the timestamp column
def clean_timestamp(timestamp):
    """
    Clean timestamp strings - handles both mm/dd/yyyy and mm-dd-yyyy formats
    Extracts only the date part (before the space)
    """
    if pd.isna(timestamp):
        return None
    
    # Extract date part (before space)
    date_part = timestamp.split(' ')[0] if ' ' in str(timestamp) else str(timestamp)
    
    # Replace dashes with slashes for uniformity
    date_part = date_part.replace('-', '/')
    
    try:
        # Parse as datetime
        return datetime.strptime(date_part, '%m/%d/%Y')
    except:
        return None

# Apply cleaning function
df['date_clean'] = df['Timestamp'].apply(clean_timestamp)

# Remove any rows where date couldn't be parsed
df_clean = df[df['date_clean'].notna()].copy()

print(f"Records after cleaning: {len(df_clean)}")
print(f"Dropped records: {len(df) - len(df_clean)}")

In [ ]:
# Count applications per day
daily_counts = df_clean.groupby('date_clean').size().reset_index(name='application_count')
daily_counts = daily_counts.sort_values('date_clean')

print(f"Number of unique days: {len(daily_counts)}")
print("\nDaily application counts:")
print(daily_counts.head(10))

In [ ]:
# Create line chart
plt.figure(figsize=(14, 6))
plt.plot(daily_counts['date_clean'], daily_counts['application_count'], 
         marker='o', linestyle='-', linewidth=2, markersize=6, color='#4285F4')

plt.title('Number of Applications Per Day', fontsize=16, fontweight='bold', pad=20)
plt.xlabel('Date', fontsize=12)
plt.ylabel('Number of Applications', fontsize=12)
plt.grid(True, alpha=0.3, linestyle='--')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()

# Add some statistics
total_apps = daily_counts['application_count'].sum()
avg_apps = daily_counts['application_count'].mean()
max_apps = daily_counts['application_count'].max()

stats_text = f'Total: {total_apps} | Avg/day: {avg_apps:.1f} | Peak: {max_apps}'
plt.text(1, 0.98, stats_text, transform=plt.gcf().transFigure, 
         ha='center', va='top', fontsize=10, bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.show()

# Close database connection
conn.close()